In [19]:
import subprocess, json, requests, re
from datetime import datetime

class ModelChecker:
    def __init__(self):
        self.models = {}
        self.active = None
        self.history = []
        self._discover()
    
    def _discover(self):
        """Find all models from Ollama."""
        try:
            # Use plain text output (more reliable than --format json)
            result = subprocess.run(
                ["ollama", "list"],
                capture_output=True, text=True, timeout=10
            )
            
            if result.returncode == 0 and result.stdout.strip():
                lines = result.stdout.strip().split('\n')
                # Skip header line
                for line in lines[1:]:
                    if line.strip():
                        parts = line.split()
                        if len(parts) >= 1:
                            name = parts[0].replace(':latest', '')
                            self.models[name] = {
                                "name": name,
                                "index": len(self.models) + 1,
                                "finetuned": 'aegis' in name.lower() or 'finetuned' in name.lower(),
                                "size": parts[2] if len(parts) > 2 else "?",
                                "modified": ' '.join(parts[3:]) if len(parts) > 3 else "?"
                            }
        except Exception as e:
            print(f"⚠️  Ollama check: {e}")
        
        if not self.models:
            print("❌ No models found. Is Ollama running?")
    
    def list(self, finetuned_only=False):
        """Display all models."""
        print("\n" + "="*70)
        print("📦 MODELS")
        print("="*70)
        
        items = {k:v for k,v in self.models.items() 
                if not finetuned_only or v["finetuned"]}
        
        if not items:
            print("No models found.")
            return
        
        # Separate fine-tuned and base
        ft_models = {k:v for k,v in items.items() if v["finetuned"]}
        base_models = {k:v for k,v in items.items() if not v["finetuned"]}
        
        if ft_models:
            print("\n🎯 FINE-TUNED MODELS:")
            for name, info in sorted(ft_models.items()):
                active = " ← ACTIVE" if name == self.active else ""
                print(f"  [{info['index']}] 🎯 {name}{active}")
                print(f"      Size: {info['size']} | Modified: {info['modified']}")
        
        if base_models:
            print("\n📦 BASE MODELS:")
            for name, info in sorted(base_models.items()):
                active = " ← ACTIVE" if name == self.active else ""
                print(f"  [{info['index']}] 📦 {name}{active}")
        
        print("="*70 + "\n")
    
    def use(self, identifier):
        """Select model by name or index number."""
        if isinstance(identifier, int):
            for name, info in self.models.items():
                if info["index"] == identifier:
                    self.active = name
                    print(f"✅ Active: 🎯 {name}")
                    return
        
        if identifier in self.models:
            self.active = identifier
            print(f"✅ Active: 🎯 {identifier}")
            return
        
        # Partial match
        matches = [n for n in self.models if identifier.lower() in n.lower()]
        if len(matches) == 1:
            self.active = matches[0]
            print(f"✅ Active: 🎯 {matches[0]}")
        elif matches:
            print(f"⚠️  Multiple matches. Be more specific:")
            for m in matches:
                print(f"   - {m}")
        else:
            print(f"❌ Not found: {identifier}")
    
    def ask(self, prompt, model=None):
        """Send query to model and print response."""
        target = model or self.active
        if not target:
            print("❌ No model selected. Use: checker.use('name') or checker.use(1)")
            return None
        
        print(f"\n{'─'*60}")
        print(f"🤖 Model: {target}")
        print(f"📝 Query: {prompt}")
        print(f"{'─'*60}")
        print("⏳ Thinking...")
        
        try:
            resp = requests.post(
                "http://localhost:11434/api/generate",
                json={
                    "model": target,
                    "prompt": f"[SRE Expert] Provide a concise, technical answer:\n\n{prompt}",
                    "stream": False,
                    "options": {"temperature": 0.7, "num_predict": 400}
                },
                timeout=120
            )
            if resp.status_code == 200:
                result = resp.json().get("response", "No response")
                self.history.append({
                    "model": target, "prompt": prompt,
                    "response": result, "time": datetime.now().strftime("%H:%M:%S")
                })
                print(f"\n{result}")
                print(f"{'─'*60}\n")
                return result
            else:
                print(f"❌ HTTP {resp.status_code}: {resp.text[:100]}")
        except requests.exceptions.ConnectionError:
            print("❌ Ollama not running. Start with: ollama serve")
        except Exception as e:
            print(f"❌ {e}")
        return None
    
    def compare(self, prompt, limit=3):
        """Compare fine-tuned models + base llama3."""
        finetuned = [(k, v) for k, v in self.models.items() if v["finetuned"]]
        finetuned = sorted(finetuned, key=lambda x: x[1]['index'])[:limit]
        
        models_to_test = [m[0] for m in finetuned]
        if "llama3" in self.models and "llama3" not in models_to_test:
            models_to_test.append("llama3")
        
        if not models_to_test:
            print("❌ No fine-tuned models to compare")
            return
        
        print(f"\n{'='*60}")
        print(f"🏟️  MODEL COMPARISON ARENA")
        print(f"📝 Query: {prompt}")
        print(f"Models: {', '.join(models_to_test)}")
        print(f"{'='*60}")
        
        results = {}
        for model in models_to_test:
            print(f"\n{'─'*60}")
            print(f"🤖 {model}")
            print(f"{'─'*60}")
            result = self.ask(prompt, model)
            results[model] = result
        
        print(f"\n{'='*60}")
        print("📊 SUMMARY")
        for model, result in results.items():
            words = len(result.split()) if result else 0
            finetuned_badge = "🎯" if self.models.get(model, {}).get("finetuned") else "📦"
            print(f"  {finetuned_badge} {model}: {words} words")
        print()
    
    def show_history(self):
        """Show last 10 queries."""
        if not self.history:
            print("No queries yet.")
            return
        print("\n📜 QUERY HISTORY")
        print("="*60)
        for i, h in enumerate(self.history[-10:], 1):
            print(f"  [{i}] {h['time']} | {h['model']}")
            print(f"      Q: {h['prompt'][:80]}...")
        print()

# ═══ CREATE INSTANCE ═══
checker = ModelChecker()

# ═══ SHOW AVAILABLE MODELS ═══
checker.list()

# ═══ QUICK HELP ═══
print("💡 COMMANDS:")
print("  checker.list()            - Show all models")
print("  checker.list(True)        - Show only fine-tuned")
print("  checker.use(1)            - Select by index")
print("  checker.use('aegis')      - Select by partial name")
print("  checker.ask('question')   - Query active model")
print("  checker.compare('query')  - Compare all fine-tuned models")
print("  checker.show_history()    - Show query history")
print()

# ═══ AUTO-SELECT FIRST FINETUNED MODEL ═══
finetuned = [k for k, v in checker.models.items() if v["finetuned"]]
if finetuned:
    checker.use(finetuned[0])
    print(f"🚀 Ready! Try:")
    print(f"   checker.ask('What causes nginx crashes?')\n")
    print(f"   checker.compare('How to handle database timeouts?')\n")
else:
    print("⚠️  No fine-tuned models found.\n")


📦 MODELS

🎯 FINE-TUNED MODELS:
  [4] 🎯 aegis-sre-deepseek-r1:7b-finetuned
      Size: 4.7 | Modified: GB 3 hours ago
  [1] 🎯 aegis-sre-llama3-all
      Size: 4.7 | Modified: GB 40 minutes ago
  [3] 🎯 aegis-sre-llama3-finetuned
      Size: 4.7 | Modified: GB 2 hours ago
  [2] 🎯 aegis-sre-llama3-user_8
      Size: 4.7 | Modified: GB 51 minutes ago

📦 BASE MODELS:
  [6] 📦 deepseek-r1:7b
  [8] 📦 llama3
  [7] 📦 llava:7b
  [5] 📦 mistral:7b

💡 COMMANDS:
  checker.list()            - Show all models
  checker.list(True)        - Show only fine-tuned
  checker.use(1)            - Select by index
  checker.use('aegis')      - Select by partial name
  checker.ask('question')   - Query active model
  checker.compare('query')  - Compare all fine-tuned models
  checker.show_history()    - Show query history

✅ Active: 🎯 aegis-sre-llama3-all
🚀 Ready! Try:
   checker.ask('What causes nginx crashes?')

   checker.compare('How to handle database timeouts?')



In [21]:
checker.use(1)

✅ Active: 🎯 aegis-sre-llama3-all


In [22]:
checker.ask("What caused ngnix server crash and how many incidents are there total ?")


────────────────────────────────────────────────────────────
🤖 Model: aegis-sre-llama3-all
📝 Query: What caused ngnix server crash and how many incidents are there total ?
────────────────────────────────────────────────────────────
⏳ Thinking...

Based on the available logs and monitoring data, I've identified the root cause of the Nginx server crash as:

**Cause:** High CPU utilization due to an unexpected surge in incoming requests, likely triggered by a sudden increase in traffic or a misconfigured load balancer. This caused the Nginx process to consume excessive memory and eventually crash.

**Incident Count:** After analyzing the log data, I've identified **3 separate incidents** that occurred within the past 24 hours, all resulting from the same underlying cause:

1. Initial crash at 02:00 UTC
2. Re-occurrence at 06:45 UTC (likely due to a lingering configuration issue)
3. Third and final incident at 12:15 UTC (after a brief window of stability)

**Remediation Steps:** To preve

"Based on the available logs and monitoring data, I've identified the root cause of the Nginx server crash as:\n\n**Cause:** High CPU utilization due to an unexpected surge in incoming requests, likely triggered by a sudden increase in traffic or a misconfigured load balancer. This caused the Nginx process to consume excessive memory and eventually crash.\n\n**Incident Count:** After analyzing the log data, I've identified **3 separate incidents** that occurred within the past 24 hours, all resulting from the same underlying cause:\n\n1. Initial crash at 02:00 UTC\n2. Re-occurrence at 06:45 UTC (likely due to a lingering configuration issue)\n3. Third and final incident at 12:15 UTC (after a brief window of stability)\n\n**Remediation Steps:** To prevent future crashes, I recommend:\n\n1. Implementing rate limiting or caching mechanisms to mitigate sudden spikes in traffic.\n2. Conducting regular load testing to identify potential issues before they impact production.\n3. Improving mon

In [23]:
checker.compare("query")


🏟️  MODEL COMPARISON ARENA
📝 Query: query
Models: aegis-sre-llama3-all, aegis-sre-llama3-user_8, aegis-sre-llama3-finetuned, llama3

────────────────────────────────────────────────────────────
🤖 aegis-sre-llama3-all
────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────
🤖 Model: aegis-sre-llama3-all
📝 Query: query
────────────────────────────────────────────────────────────
⏳ Thinking...

I'm happy to help! However, I need more context to provide an accurate response. Can you please provide more details about the query you're referring to? Is it related to an incident, error message, or perhaps a system performance issue? The more information you can provide, the better I'll be able to assist you with root cause analysis and remediation steps.
────────────────────────────────────────────────────────────


────────────────────────────────────────────────────────────
🤖 aegis-sre-llama3-user_8
──────────────────────────

In [24]:
checker.show_history()


📜 QUERY HISTORY
  [1] 05:18:49 | aegis-sre-llama3-all
      Q: What caused ngnix server crash and how many incidents are there total ?...
  [2] 05:19:42 | aegis-sre-llama3-all
      Q: query...
  [3] 05:19:44 | aegis-sre-llama3-user_8
      Q: query...
  [4] 05:19:45 | aegis-sre-llama3-finetuned
      Q: query...
  [5] 05:19:47 | llama3
      Q: query...



In [25]:
checker.use(4)

✅ Active: 🎯 aegis-sre-deepseek-r1:7b-finetuned


In [26]:
checker.ask("Hello")


────────────────────────────────────────────────────────────
🤖 Model: aegis-sre-deepseek-r1:7b-finetuned
📝 Query: Hello
────────────────────────────────────────────────────────────
⏳ Thinking...

Hello! How can I assist you today?
────────────────────────────────────────────────────────────



'Hello! How can I assist you today?'